In [3]:
!pip install langchain langchain-core langchain-community chromadb sentence-transformers pypdf transformers accelerate huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 65.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.2/331.2 kB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.0/220.0 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/10

In [4]:
from huggingface_hub import login
# This will prompt for your HF token
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
model_id = "google/gemma-2b-it"

In [6]:
tokenizer = AutoTokenizer.from_pretrained(model_id)

config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/34.2k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

In [7]:
model = AutoModelForCausalLM.from_pretrained(
model_id,
torch_dtype=torch.float32,
device_map="auto"
)

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/13.5k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

In [27]:
from google.colab import files

uploaded = files.upload()

file_paths = []
for filename, content in uploaded.items():
    with open(filename, "wb") as f:
        f.write(content)
    file_paths.append(filename)

print(f"Uploaded files: {file_paths}")

Saving ISO_IEC-270012022-ed.3.pdf to ISO_IEC-270012022-ed.3 (1).pdf
Saving NIST.SP.800-53r5.pdf to NIST.SP.800-53r5 (1).pdf
Uploaded files: ['ISO_IEC-270012022-ed.3 (1).pdf', 'NIST.SP.800-53r5 (1).pdf']


In [28]:
from langchain_community.document_loaders import PyPDFLoader

docs = []
for path in file_paths:
    loader = PyPDFLoader(path)
    docs.extend(loader.load())

print(f"Total documents loaded: {len(docs)}")

Total documents loaded: 518


In [10]:
print(f"{docs[0].page_content[:200]}\n")
print(docs[0].metadata)

Information security, cybersecurity 
and privacy protection — Information 
security management systems — 
Requirements
Sécurité de l'information, cybersécurité et protection de la vie 
privée — Systèm

{'producer': 'Adobe PDF Library 16.0; modified using iText® 7.1.12 ©2000-2020 iText Group NV (AGPL-version)', 'creator': 'Adobe InDesign 16.4 (Windows)', 'creationdate': '2022-09-29T17:54:00+02:00', 'author': 'ISO', 'moddate': '2022-10-31T13:57:54+01:00', 'title': '\ufeffISO/IEC 27001:2022\ufeff', 'trapped': '/False', 'source': 'ISO_IEC-270012022-ed.3.pdf', 'total_pages': 26, 'page': 0, 'page_label': '1'}


In [11]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, add_start_index=True
)
all_splits = text_splitter.split_documents(docs)


print(len(all_splits))

81


In [12]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embed = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

/tmp/ipython-input-3795205697.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embed = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [13]:
from langchain_community.vectorstores import Chroma

db = Chroma.from_documents(all_splits, embed)
retriever = db.as_retriever()

In [14]:
from langchain_core.language_models import LLM
from langchain_core.outputs import Generation, LLMResult
from langchain_core.prompts import PromptTemplate
from typing import ClassVar


class LocalLLM(LLM):
    def _call(self, prompt: str, stop=None) -> str:
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=True,
            temperature=0.2
        )
        return tokenizer.decode(outputs[0], skip_special_tokens=True)
    @property
    def _identifying_params(self):
        return {"name": "gemma-local"}
    @property
    def _llm_type(self):
        return "custom"
    reasoning_prompt: ClassVar[PromptTemplate] = PromptTemplate(
    input_variables=["context", "question"],
    template="""
You are an expert reasoning assistant.
Use the retrieved context to answer the question step-by-step.
First, think through the problem logically.
Then, provide the final concise answer.

Context:
{context}

Question:
{question}

Reasoning:
1. Identify relevant facts from the context.
2. Connect the facts logically.
3. Derive the answer.

Final Answer:
"""
)

In [15]:
from langchain_classic.chains import RetrievalQA

llm = LocalLLM()

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=db.as_retriever(search_kwargs={"k": 3}),
    chain_type="stuff",
    chain_type_kwargs={"prompt": llm.reasoning_prompt}
)

In [ ]:
question = "Generate a sample organizational cybersecurity policy for a finance company to deal with company owned devices aligned with NIST Special Publication 800-53. Generate 10 policy suggestions."
response = qa_chain.invoke(question)
print(response['result'])


# Task
The RAG model now explicitly offers reasoning alongside its retrieval-based answers, making it more 'agentic' by demonstrating its thought process.

## Modify Reasoning Prompt

### Subtask:
Refine the `reasoning_prompt` within the `LocalLLM` class to instruct the model to explicitly state its step-by-step reasoning process before providing the final answer. This will ensure that the reasoning steps (Identify facts, Connect facts, Derive answer) are generated and displayed.


**Reasoning**:
The current `reasoning_prompt` instructs the model to repeat the reasoning steps rather than generating its own. I need to modify the prompt template to explicitly ask the model to output its reasoning process based on the given steps, before providing the final answer.



In [ ]:
from langchain_core.language_models import LLM
from langchain_core.outputs import Generation, LLMResult
from langchain_core.prompts import PromptTemplate
from typing import ClassVar


class LocalLLM(LLM):
    def _call(self, prompt: str, stop=None) -> str:
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=True,
            temperature=0.2
        )
        return tokenizer.decode(outputs[0], skip_special_tokens=True)
    @property
    def _identifying_params(self):
        return {"name": "gemma-local"}
    @property
    def _llm_type(self):
        return "custom"
    reasoning_prompt: ClassVar[PromptTemplate] = PromptTemplate(
    input_variables=["context", "question"],
    template="""
You are an expert reasoning assistant. Your goal is to answer the question thoroughly by first outlining your step-by-step reasoning process, and then providing a concise final answer.

Instructions for Reasoning:
1. Identify relevant facts from the provided context.
2. Connect these facts logically to address the question.
3. Derive the final answer based on the connected facts.

Context:
{context}

Question:
{question}

Your Step-by-Step Reasoning:
"""
)


**Reasoning**:
The previous code block for the `LocalLLM` class defined the `reasoning_prompt` but did not explicitly include a 'Final Answer:' section after the 'Your Step-by-Step Reasoning:' section as required by the instructions. I will modify the template to add this section, ensuring the model is explicitly instructed to provide the final answer after its reasoning.



In [ ]:
from langchain_core.language_models import LLM
from langchain_core.outputs import Generation, LLMResult
from langchain_core.prompts import PromptTemplate
from typing import ClassVar


class LocalLLM(LLM):
    def _call(self, prompt: str, stop=None) -> str:
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=True,
            temperature=0.2
        )
        return tokenizer.decode(outputs[0], skip_special_tokens=True)
    @property
    def _identifying_params(self):
        return {"name": "gemma-local"}
    @property
    def _llm_type(self):
        return "custom"
    reasoning_prompt: ClassVar[PromptTemplate] = PromptTemplate(
    input_variables=["context", "question"],
    template="""
You are an expert reasoning assistant. Your goal is to answer the question thoroughly by first outlining your step-by-step reasoning process, and then providing a concise final answer.

Instructions for Reasoning:
1. Identify relevant facts from the provided context.
2. Connect these facts logically to address the question.
3. Derive the final answer based on the connected facts.

Context:
{context}

Question:
{question}

Your Step-by-Step Reasoning:

Final Answer:
"""
)


In [ ]:
llm = LocalLLM()

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=db.as_retriever(search_kwargs={"k": 3}),
    chain_type="stuff",
    chain_type_kwargs={"prompt": llm.reasoning_prompt}
)

question = "Generate a sample organizational cybersecurity policy for a finance company to deal with company owned devices aligned with NIST Special Publication 800-53. Generate 10 policy suggestions."
response = qa_chain.invoke(question)
print(response['result'])


You are an expert reasoning assistant. Your goal is to answer the question thoroughly by first outlining your step-by-step reasoning process, and then providing a concise final answer.

Instructions for Reasoning:
1. Identify relevant facts from the provided context.
2. Connect these facts logically to address the question.
3. Derive the final answer based on the connected facts.

Context:
NIST Special Publication 800-53 
Revision 5 
 
 
 Security and Privacy Controls for 
Information Systems and Organizations                                                                                                
 
 
 
JOINT TASK FORCE 
 
 
 
 
 
 
 
This publication is available free of charge from: 
https://doi.org/10.6028/NIST.SP.800-53r5  
 
 
 
 
September 2020 
INCLUDES UPDATES AS OF 12-10-2020; SEE PAGE XVII 
 
 
 
 
 
 
 
 
 
 
 
U.S. Department of Commerce  
Wilbur L. Ross, Jr., Secretary 
 
National Institute of Standards and Technology  
      Walter Copan, NIST Director and Under S

In [ ]:
from langchain_classic.chains import RetrievalQA

llm = LocalLLM()

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=db.as_retriever(search_kwargs={"k": 3}),
    chain_type="stuff",
    chain_type_kwargs={"prompt": llm.reasoning_prompt}
)

question = "Generate a sample organizational cybersecurity policy for a finance company to deal with company owned devices aligned with NIST Special Publication 800-53. Generate 10 policy suggestions."
response = qa_chain.invoke(question)
print(response['result'])


You are an expert reasoning assistant. Your goal is to answer the question thoroughly by first outlining your step-by-step reasoning process, and then providing a concise final answer.

Instructions for Reasoning:
1. Identify relevant facts from the provided context.
2. Connect these facts logically to address the question.
3. Derive the final answer based on the connected facts.

Context:
NIST Special Publication 800-53 
Revision 5 
 
 
 Security and Privacy Controls for 
Information Systems and Organizations                                                                                                
 
 
 
JOINT TASK FORCE 
 
 
 
 
 
 
 
This publication is available free of charge from: 
https://doi.org/10.6028/NIST.SP.800-53r5  
 
 
 
 
September 2020 
INCLUDES UPDATES AS OF 12-10-2020; SEE PAGE XVII 
 
 
 
 
 
 
 
 
 
 
 
U.S. Department of Commerce  
Wilbur L. Ross, Jr., Secretary 
 
National Institute of Standards and Technology  
      Walter Copan, NIST Director and Under S

In [1]:
pip install rank_bm25

In [17]:
from langchain_community.retrievers.bm25 import BM25Retriever

# The BM25Retriever.from_texts method expects raw text strings, which it tokenizes internally.
# So, we extract the page_content as is.
tokenized_docs = [doc.page_content for doc in all_splits]

bm25_retriever = BM25Retriever.from_texts(tokenized_docs)
print("BM25 Retriever created successfully.")

BM25 Retriever created successfully.


In [20]:
from typing import List
from langchain_core.documents import Document
from langchain_core.retrievers import BaseRetriever
from langchain_core.callbacks import CallbackManagerForRetrieverRun
from pydantic import Field

# Define the RRF function
def reciprocal_rank_fusion(results: List[List[Document]], a: float = 0.4) -> List[Document]:
    """
    Applies Reciprocal Rank Fusion to a list of lists of documents.

    Args:
        results: A list of lists of Document objects, where each inner list
                 comes from a different retriever.
        a: The 'a' parameter for RRF, controlling the influence of lower ranks.

    Returns:
        A single list of Document objects, re-ranked by RRF score.
    """
    fused_scores = {}
    doc_map = {}

    # Collect all unique documents and map their page_content to the Document object
    for doc_list in results:
        for doc in doc_list:
            if doc.page_content not in doc_map:
                doc_map[doc.page_content] = doc

    # Calculate fused scores
    for doc_list in results:
        for rank, doc in enumerate(doc_list):
            doc_id = doc.page_content
            # Ensure score is initialized if a document appears only in later lists
            if doc_id not in fused_scores:
                fused_scores[doc_id] = 0
            fused_scores[doc_id] += 1 / (a + rank + 1)

    # Sort documents by fused score in descending order
    reranked_docs = sorted(
        doc_map.values(),
        key=lambda doc: fused_scores.get(doc.page_content, 0),
        reverse=True
    )

    return reranked_docs


class RRFHybridRetriever(BaseRetriever):
    """Retriever that combines multiple retrievers using reciprocal rank fusion."""

    retrievers: List[BaseRetriever] = Field(..., description="List of retrievers to combine.")
    rrf_a: float = Field(default=0.4, description="The 'a' parameter for Reciprocal Rank Fusion.")
    k_merge: int = Field(default=10, description="The number of documents to return after fusion.")

    class Config:
        arbitrary_types_allowed = True

    def _get_relevant_documents(
        self, query: str, *, run_manager: CallbackManagerForRetrieverRun = None
    ) -> List[Document]:
        retrieved_docs_lists = []
        for r in self.retrievers:
            # Use run_manager if provided, otherwise call directly
            retrieved_docs_lists.append(r.get_relevant_documents(query, run_manager=run_manager))

        fused_docs = reciprocal_rank_fusion(retrieved_docs_lists, a=self.rrf_a)
        return fused_docs[:self.k_merge]

# Create a list of the retrievers we want to combine
# 'retriever' is the Chroma DB retriever, 'bm25_retriever' is the BM25 retriever
retrievers_to_combine = [bm25_retriever, retriever]

# Create the RRFHybridRetriever instance
hybrid_retriever = RRFHybridRetriever(
    retrievers=retrievers_to_combine,
    rrf_a=0.4 # Using the specified 'a' parameter
)

print("Hybrid retriever (BM25 + Chroma with RRF) created successfully.")

Hybrid retriever (BM25 + Chroma with RRF) created successfully.


/tmp/ipython-input-2354038079.py:48: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  class RRFHybridRetriever(BaseRetriever):


In [22]:
from typing import List
from langchain_core.documents import Document
from langchain_core.retrievers import BaseRetriever
from langchain_core.callbacks import CallbackManagerForRetrieverRun
from pydantic import Field

# Define the RRF function
def reciprocal_rank_fusion(results: List[List[Document]], a: float = 0.4) -> List[Document]:
    """
    Applies Reciprocal Rank Fusion to a list of lists of documents.

    Args:
        results: A list of lists of Document objects, where each inner list
                 comes from a different retriever.
        a: The 'a' parameter for RRF, controlling the influence of lower ranks.

    Returns:
        A single list of Document objects, re-ranked by RRF score.
    """
    fused_scores = {}
    doc_map = {}

    # Collect all unique documents and map their page_content to the Document object
    for doc_list in results:
        for doc in doc_list:
            if doc.page_content not in doc_map:
                doc_map[doc.page_content] = doc

    # Calculate fused scores
    for doc_list in results:
        for rank, doc in enumerate(doc_list):
            doc_id = doc.page_content
            # Ensure score is initialized if a document appears only in later lists
            if doc_id not in fused_scores:
                fused_scores[doc_id] = 0
            fused_scores[doc_id] += 1 / (a + rank + 1)

    # Sort documents by fused score in descending order
    reranked_docs = sorted(
        doc_map.values(),
        key=lambda doc: fused_scores.get(doc.page_content, 0),
        reverse=True
    )

    return reranked_docs


class RRFHybridRetriever(BaseRetriever):
    """Retriever that combines multiple retrievers using reciprocal rank fusion."""

    retrievers: List[BaseRetriever] = Field(..., description="List of retrievers to combine.")
    rrf_a: float = Field(default=0.4, description="The 'a' parameter for Reciprocal Rank Fusion.")
    k_merge: int = Field(default=10, description="The number of documents to return after fusion.")

    class Config:
        arbitrary_types_allowed = True

    def _get_relevant_documents(
        self, query: str, *, run_manager: CallbackManagerForRetrieverRun = None
    ) -> List[Document]:
        retrieved_docs_lists = []
        for r in self.retrievers:
            # Call invoke method, which typically delegates to _get_relevant_documents
            # and handles callbacks.
            retrieved_docs_lists.append(r.invoke(query, config={"callbacks": run_manager.get_child() if run_manager else None}))

        fused_docs = reciprocal_rank_fusion(retrieved_docs_lists, a=self.rrf_a)
        return fused_docs[:self.k_merge]

# Create a list of the retrievers we want to combine
# 'retriever' is the Chroma DB retriever, 'bm25_retriever' is the BM25 retriever
retrievers_to_combine = [bm25_retriever, retriever]

# Create the RRFHybridRetriever instance
hybrid_retriever = RRFHybridRetriever(
    retrievers=retrievers_to_combine,
    rrf_a=0.4 # Using the specified 'a' parameter
)

print("Hybrid retriever (BM25 + Chroma with RRF) created successfully.")

llm = LocalLLM()

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=hybrid_retriever,
    chain_type="stuff",
    chain_type_kwargs={"prompt": llm.reasoning_prompt}
)

question = "Generate a sample organizational cybersecurity policy for a finance company to deal with company owned devices aligned with NIST Special Publication 800-53. Generate 10 policy suggestions."
response = qa_chain.invoke(question)
print(response['result'])

/tmp/ipython-input-1850377359.py:48: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  class RRFHybridRetriever(BaseRetriever):


Hybrid retriever (BM25 + Chroma with RRF) created successfully.

You are an expert reasoning assistant.
Use the retrieved context to answer the question step-by-step.
First, think through the problem logically.
Then, provide the final concise answer.

Context:
ISO/IEC 27001:2022(E)
5.2  Policy
Top management shall establish an information security policy that:
a) is appropriate to the purpose of the organization;
b) includes information security objectives (see 6.2) or provides the framework for setting information 
security objectives;
c) includes a commitment to satisfy applicable requirements related to information security; 
d) includes a commitment to continual improvement of the information security management system.
The information security policy shall:
e) be available as documented information;
f) be communicated within the organization; 
g) be available to interested parties, as appropriate.
5.3  Organizational roles, responsibilities and authorities
Top management shall ens

**Reasoning**:
The previous code produced a `PydanticDeprecatedSince20` warning due to the deprecated `class Config` within the `RRFHybridRetriever`. I need to update this to use `ConfigDict` as recommended by Pydantic v2 to eliminate the warning.



In [25]:
from typing import List
from langchain_core.documents import Document
from langchain_core.retrievers import BaseRetriever
from langchain_core.callbacks import CallbackManagerForRetrieverRun
from pydantic import Field, ConfigDict

# Define the RRF function
def reciprocal_rank_fusion(results: List[List[Document]], a: float = 0.4) -> List[Document]:
    """
    Applies Reciprocal Rank Fusion to a list of lists of documents.

    Args:
        results: A list of lists of Document objects, where each inner list
                 comes from a different retriever.
        a: The 'a' parameter for RRF, controlling the influence of lower ranks.

    Returns:
        A single list of Document objects, re-ranked by RRF score.
    """
    fused_scores = {}
    doc_map = {}

    # Collect all unique documents and map their page_content to the Document object
    for doc_list in results:
        for doc in doc_list:
            if doc.page_content not in doc_map:
                doc_map[doc.page_content] = doc

    # Calculate fused scores
    for doc_list in results:
        for rank, doc in enumerate(doc_list):
            doc_id = doc.page_content
            # Ensure score is initialized if a document appears only in later lists
            if doc_id not in fused_scores:
                fused_scores[doc_id] = 0
            fused_scores[doc_id] += 1 / (a + rank + 1)

    # Sort documents by fused score in descending order
    reranked_docs = sorted(
        doc_map.values(),
        key=lambda doc: fused_scores.get(doc.page_content, 0),
        reverse=True
    )

    return reranked_docs


class RRFHybridRetriever(BaseRetriever):
    """Retriever that combines multiple retrievers using reciprocal rank fusion."""

    retrievers: List[BaseRetriever] = Field(..., description="List of retrievers to combine.")
    rrf_a: float = Field(default=0.4, description="The 'a' parameter for Reciprocal Rank Fusion.")
    k_merge: int = Field(default=10, description="The number of documents to return after fusion.")

    model_config = ConfigDict(arbitrary_types_allowed=True)

    def _get_relevant_documents(
        self, query: str, *, run_manager: CallbackManagerForRetrieverRun = None
    ) -> List[Document]:
        retrieved_docs_lists = []
        for r in self.retrievers:
            # Call invoke method, which typically delegates to _get_relevant_documents
            # and handles callbacks.
            retrieved_docs_lists.append(r.invoke(query, config={"callbacks": run_manager.get_child() if run_manager else None}))

        fused_docs = reciprocal_rank_fusion(retrieved_docs_lists, a=self.rrf_a)
        return fused_docs[:self.k_merge]

# Create a list of the retrievers we want to combine
# 'retriever' is the Chroma DB retriever, 'bm25_retriever' is the BM25 retriever
retrievers_to_combine = [bm25_retriever, retriever]

# Create the RRFHybridRetriever instance
hybrid_retriever = RRFHybridRetriever(
    retrievers=retrievers_to_combine,
    rrf_a=0.4 # Using the specified 'a' parameter
)

print("Hybrid retriever (BM25 + Chroma with RRF) created successfully.")

llm = LocalLLM()

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=hybrid_retriever,
    chain_type="stuff",
    chain_type_kwargs={"prompt": llm.reasoning_prompt}
)

question = "Generate a sample organizational cybersecurity policy for a finance company to deal with company owned devices aligned with ISO 27001. Generate 10 policy suggestions."
response = qa_chain.invoke(question)
print(response['result'])

Hybrid retriever (BM25 + Chroma with RRF) created successfully.

You are an expert reasoning assistant.
Use the retrieved context to answer the question step-by-step.
First, think through the problem logically.
Then, provide the final concise answer.

Context:
ISO/IEC 27001:2022(E)
5.2  Policy
Top management shall establish an information security policy that:
a) is appropriate to the purpose of the organization;
b) includes information security objectives (see 6.2) or provides the framework for setting information 
security objectives;
c) includes a commitment to satisfy applicable requirements related to information security; 
d) includes a commitment to continual improvement of the information security management system.
The information security policy shall:
e) be available as documented information;
f) be communicated within the organization; 
g) be available to interested parties, as appropriate.
5.3  Organizational roles, responsibilities and authorities
Top management shall ens

In [45]:
from typing import List
from langchain_core.documents import Document
from langchain_core.retrievers import BaseRetriever
from langchain_core.callbacks import CallbackManagerForRetrieverRun
from pydantic import Field, ConfigDict

# Define the RRF function
def reciprocal_rank_fusion(results: List[List[Document]], a: float = 0.4) -> List[Document]:
    """
    Applies Reciprocal Rank Fusion to a list of lists of documents.

    Args:
        results: A list of lists of Document objects, where each inner list
                 comes from a different retriever.
        a: The 'a' parameter for RRF, controlling the influence of lower ranks.

    Returns:
        A single list of Document objects, re-ranked by RRF score.
    """
    fused_scores = {}
    doc_map = {}

    # Collect all unique documents and map their page_content to the Document object
    for doc_list in results:
        for doc in doc_list:
            if doc.page_content not in doc_map:
                doc_map[doc.page_content] = doc

    # Calculate fused scores
    for doc_list in results:
        for rank, doc in enumerate(doc_list):
            doc_id = doc.page_content
            # Ensure score is initialized if a document appears only in later lists
            if doc_id not in fused_scores:
                fused_scores[doc_id] = 0
            fused_scores[doc_id] += 1 / (a + rank + 1)

    # Sort documents by fused score in descending order
    reranked_docs = sorted(
        doc_map.values(),
        key=lambda doc: fused_scores.get(doc.page_content, 0),
        reverse=True
    )

    return reranked_docs


class RRFHybridRetriever(BaseRetriever):
    """Retriever that combines multiple retrievers using reciprocal rank fusion."""

    retrievers: List[BaseRetriever] = Field(..., description="List of retrievers to combine.")
    rrf_a: float = Field(default=0.4, description="The 'a' parameter for Reciprocal Rank Fusion.")
    k_merge: int = Field(default=10, description="The number of documents to return after fusion.")

    model_config = ConfigDict(arbitrary_types_allowed=True)

    def _get_relevant_documents(
        self, query: str, *, run_manager: CallbackManagerForRetrieverRun = None
    ) -> List[Document]:
        retrieved_docs_lists = []
        for r in self.retrievers:
            # Call invoke method, which typically delegates to _get_relevant_documents
            # and handles callbacks.
            retrieved_docs_lists.append(r.invoke(query, config={"callbacks": run_manager.get_child() if run_manager else None}))

        fused_docs = reciprocal_rank_fusion(retrieved_docs_lists, a=self.rrf_a)
        return fused_docs[:self.k_merge]

# Create a list of the retrievers we want to combine
# 'retriever' is the Chroma DB retriever, 'bm25_retriever' is the BM25 retriever
retrievers_to_combine = [bm25_retriever, retriever]

# Create the RRFHybridRetriever instance
hybrid_retriever = RRFHybridRetriever(
    retrievers=retrievers_to_combine,
    rrf_a=0.4 # Using the specified 'a' parameter
)

print("Hybrid retriever (BM25 + Chroma with RRF) created successfully.")

llm = LocalLLM()

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=hybrid_retriever,
    chain_type="stuff",
    chain_type_kwargs={"prompt": llm.reasoning_prompt}
)

Hybrid retriever (BM25 + Chroma with RRF) created successfully.
